# Lesson 8 : Human-in-the-loop (HITL)

The participant in agent is not only a computer agent or a program code, but also a human can be involved in the conversation - i.e., in the agentic loop.

Using Agent Framework, you can apply either of the following patterns to implement human involving (such as, approval request) in agentic loop.

- You can handle human input in local tool's calling. Especially in Agent Framework, you can specify ```@ai_function(approval_mode="always_require")``` decorator in function definition, and you can process the human-in-the-loop separately from the main logic. Please see [official document](https://learn.microsoft.com/en-us/agent-framework/tutorials/agents/function-tools-approvals?pivots=programming-language-python) for this implementation.
- In workflows, you can invoke ```request_info()``` method of workflow context and you can handle human input as an event outside of your workflow. This pattern can integrate with checkpointing in workflows (see [Lesson 7](./07_workflow.ipynb) for workflow's chekpoint), and you can then handle human-in-the-loop in long running (taking 2-3 days or more) process.

In this example, we briefly explore the implementation of human-in-the-loop in the latter approach.

> Note : This example uses generic ```WorkflowBuilder```, but in high-level workflow builders (```SequentialBuilder```, ```ConcurrentBuilder```, ```GroupChatBuilder```, or ```HandoffBuilder```), you can use ```with_request_info()```, instead. (This method calls ```request_info()``` internally.)

First we initialize the client.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(credential=credential)

Now we build a workflow with human interaction.

In this example, I have changed the example in Lesson 7 (travel planning example) to include human approval at the end.

I have only replaced ```ResponseGenerator``` executor (in Lesson 7) with new ```HumanReview``` executor.  
In this executor, we invoke ```request_info()``` (see above) in handler method.  
After the human performs approval request, the method with ```@response_handler``` decorator will be invoked.

This example briefly show the result in text, but you can also build revision loop by building a loop in the workflow's edge and invoking ```ctx.send_message()``` in response handler.

In [2]:
from dataclasses import dataclass
from agent_framework import ChatAgent, ChatMessage
from agent_framework import (
    WorkflowBuilder,
    Executor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    WorkflowContext,
    handler,
    response_handler
)

@dataclass
class ApprovalRequest:
    # this class is serialized when using checkpoint
    prompt: str = ""
    draft: str = ""

class HumanReview(Executor):
    @handler
    async def generate_human_review_request(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext
    ) -> None:
        await ctx.request_info(
            request_data=ApprovalRequest(
                prompt=(
                    "旅行計画のドラフトを確認し、'approve' または 'reject' のテキストで回答してください。"  # "review the draft and reply 'approve' or 'reject' in text."
                    "(それ以外が入力された場合、処理は却下されます。)"  # (Otherwise this draft is rejected)
                ),
                draft=response.agent_run_response.text,
            ),
            response_type=str,
        )

    @response_handler
    async def do_after_human_response(
        self,
        original_request: ApprovalRequest,
        approve_or_reject: str,
        ctx: WorkflowContext[AgentExecutorRequest | str, str],
    ) -> None:
        if approve_or_reject.strip().lower() == "approve":
            await ctx.yield_output(f"Approved ! :\n----------\n {original_request.draft}")
        else:
            await ctx.yield_output("The draft is rejected.")

workflow = (
    WorkflowBuilder()
    .register_agent(
        lambda: ChatAgent(
            name="PlanningAgent",
            chat_client=client,
            instructions="ユーザーの計画作成を支援し、箇条書きの 5 項目に簡潔にまとめます。",  # "your task is to help users plan and concisely summarize it in five bullet points."
        ),
        name="PlanningAgent",
        output_response=True,
    )
    .register_agent(
        lambda: ChatAgent(
            name="ReviseAgent",
            chat_client=client,
            instructions="与えられた計画を確認して、より洗練させます。",  # "your task is to review the plan you receive and to refine it further."
        ),
        name="ReviseAgent",
        output_response=False,
    )
    .register_executor(
        lambda: HumanReview(id="final_human_review"),
        name="FinalReview"
    )
    .add_edge("PlanningAgent", "ReviseAgent")
    .add_edge("ReviseAgent", "FinalReview")
    .set_start_executor("PlanningAgent")
    .build()
)

Now we run this workflow.

When ```request_info()``` is invoked, ```RequestInfoEvent``` is captured in streaming loop.   
Before proceeding to the next, the event loop is then suspended.

In this example, the approval response is stored in ```responses``` variable. (I always return "approve" in this example, because Jupyter notebook cannot handle interactive response.)

In [3]:
from agent_framework import RequestInfoEvent, WorkflowOutputEvent

responses: dict[str, str] = {}

async for event in workflow.run_stream("大阪の旅行計画を作成してください。"):  # "create a plan for my travel in Osaka"
    if isinstance(event, RequestInfoEvent):
        approval_request = event.data
        request_id = event.request_id

        print("----------------------------------")
        print(approval_request.prompt)
        print("----------------------------------")
        print(approval_request.draft)
        response = "approve"   # always send approve

        responses[request_id] = response

----------------------------------
旅行計画のドラフトを確認し、'approve' または 'reject' のテキストで回答してください。(それ以外が入力された場合、処理は却下されます。)
----------------------------------
- **日程・目的を先に決める**：何泊（例：1泊2日/2泊3日）・同行者・予算・目的（食べ歩き/観光/USJ/買い物）を整理し、エリアを「キタ（梅田）」と「ミナミ（難波）」で日ごとに分ける  
- **モデル行程（2泊3日例）**：1日目ミナミ（道頓堀・心斎橋・なんば）→夜景／2日目USJ（終日）or 大阪城＋中之島＋梅田／3日目新世界（通天閣）＋黒門市場→帰路  
- **必訪スポット候補**：道頓堀（川沿い散歩・グリコ）／大阪城公園／新世界・通天閣／梅田スカイビル（夕方〜夜）＋好みで海遊館・中之島美術館・天神橋筋商店街  
- **グルメ枠を固定**：たこ焼き・お好み焼き・串カツを各1回＋黒門市場/天満で食べ歩き、混雑回避に「人気店は開店前後」か「平日昼」を狙う  
- **移動・宿・予約**：地下鉄＋徒歩が基本（ICカード推奨）／宿は難波・心斎橋 or 梅田が便利／USJ・展望台・人気店は事前予約、雨天用に商業施設（なんば/梅田）も用意  

※よければ「何泊・行く月・USJ行く/行かない・誰と（家族/カップル/一人）」を教えてください。条件に合わせて時間付きの行程に落とし込みます。


Again start the streaming loop in workflow by responding approval request with ```send_responses_streaming()``` method.

The workflow has completed, and the text "Approved !" is appended in the final output.

In [4]:
from agent_framework import WorkflowOutputEvent

async for event in workflow.send_responses_streaming(responses):
    if isinstance(event, WorkflowOutputEvent):
        final_output = event.data

print(final_output)

Approved ! :
----------
 - **日程・目的を先に決める**：何泊（例：1泊2日/2泊3日）・同行者・予算・目的（食べ歩き/観光/USJ/買い物）を整理し、エリアを「キタ（梅田）」と「ミナミ（難波）」で日ごとに分ける  
- **モデル行程（2泊3日例）**：1日目ミナミ（道頓堀・心斎橋・なんば）→夜景／2日目USJ（終日）or 大阪城＋中之島＋梅田／3日目新世界（通天閣）＋黒門市場→帰路  
- **必訪スポット候補**：道頓堀（川沿い散歩・グリコ）／大阪城公園／新世界・通天閣／梅田スカイビル（夕方〜夜）＋好みで海遊館・中之島美術館・天神橋筋商店街  
- **グルメ枠を固定**：たこ焼き・お好み焼き・串カツを各1回＋黒門市場/天満で食べ歩き、混雑回避に「人気店は開店前後」か「平日昼」を狙う  
- **移動・宿・予約**：地下鉄＋徒歩が基本（ICカード推奨）／宿は難波・心斎橋 or 梅田が便利／USJ・展望台・人気店は事前予約、雨天用に商業施設（なんば/梅田）も用意  

※よければ「何泊・行く月・USJ行く/行かない・誰と（家族/カップル/一人）」を教えてください。条件に合わせて時間付きの行程に落とし込みます。
